# Phase 8 Wave 2 Dependency and Environment Check

This notebook is the controlled dependency gate for Phase 8 Wave 2 external GBDTs.
It creates a separate Wave 2 environment under `C:\tmp`, installs only the
authorized pinned packages there, writes dependency/environment reports, and leaves
the project `.venv`, dependency files, official data, submissions, and main
experiment log untouched.

Phase 9, Phase 10, and Phase 11 remain locked.


In [1]:
from __future__ import annotations

import csv
import hashlib
import importlib.metadata as metadata
import importlib.util
import os
import platform
import subprocess
import sys
import textwrap
import venv
from pathlib import Path

AUTHORIZED_HEAD = "98d8bb9bc0cc2cccd0c3722a9efebf56ab63021e"
EXPERIMENT_ID = "phase8_wave2_external_gbdt_v1"
os.environ["MPLBACKEND"] = "Agg"
REPO_ROOT = Path(r"C:\GitHub\reto_Tokio")
assert (REPO_ROOT / ".git").exists(), f"Repository marker not found: {REPO_ROOT / '.git'}"
ENV_DIR = Path(r"C:\tmp\reto_tokio_phase8_wave2_env")
ENV_PYTHON = ENV_DIR / "Scripts" / "python.exe"
DEPENDENCY_REPORT = REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_dependency_report.csv"
ENVIRONMENT_REPORT = REPO_ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_environment_report.md"
FORBIDDEN_PATHS = [
    "data/input",
    "notebooks/_official",
    "references",
    "outputs/submissions",
    "logs/experiment_log.csv",
    ".vscode/settings.json",
]

def run_cmd(args: list[str], *, cwd: Path = REPO_ROOT, check: bool = True) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"
    result = subprocess.run(args, cwd=str(cwd), text=True, capture_output=True, env=env)
    print("$", " ".join(args))
    if result.stdout.strip():
        print(result.stdout.strip()[-4000:])
    if result.stderr.strip():
        print(result.stderr.strip()[-4000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {' '.join(args)}")
    return result

def safe_write_text(path: Path, text: str) -> None:
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite artifact: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

def safe_write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite artifact: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

print("Repository root:", REPO_ROOT)
print("Base executable:", sys.executable)
print("Separate env target:", ENV_DIR)


Repository root: C:\GitHub\reto_Tokio
Base executable: C:\GitHub\reto_Tokio\.venv\Scripts\python.exe
Separate env target: C:\tmp\reto_tokio_phase8_wave2_env


## Repository and Artifact Guards


In [2]:
head = run_cmd(["git", "rev-parse", "HEAD"]).stdout.strip()
assert head == AUTHORIZED_HEAD, f"HEAD mismatch: {head}"
assert not run_cmd(["git", "diff", "--cached", "--name-only"]).stdout.strip(), "Staged files present"
run_cmd(["git", "diff", "--check"])
assert not run_cmd(["git", "diff", "--name-only", "--", *FORBIDDEN_PATHS]).stdout.strip(), "Forbidden-path diff present"
assert not run_cmd(["git", "diff", "--name-only"]).stdout.strip(), "Unexpected tracked modifications present"

if ENV_DIR.exists():
    raise FileExistsError(f"Separate Wave 2 env already exists: {ENV_DIR}")
for artifact in [DEPENDENCY_REPORT, ENVIRONMENT_REPORT]:
    if artifact.exists():
        raise FileExistsError(f"Wave 2 artifact already exists: {artifact}")

print("Repository gates: PASS")


$ git rev-parse HEAD
98d8bb9bc0cc2cccd0c3722a9efebf56ab63021e
$ git diff --cached --name-only
$ git diff --check
$ git diff --name-only -- data/input notebooks/_official references outputs/submissions logs/experiment_log.csv .vscode/settings.json


$ git diff --name-only
Repository gates: PASS


## Base Environment Snapshot


In [3]:
def dist_version(package: str) -> str:
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return "not_installed"

base_packages = {
    "python": platform.python_version(),
    "numpy": dist_version("numpy"),
    "pandas": dist_version("pandas"),
    "scikit-learn": dist_version("scikit-learn"),
    "xgboost": dist_version("xgboost"),
    "lightgbm": dist_version("lightgbm"),
    "catboost": dist_version("catboost"),
}
print(base_packages)
assert base_packages["python"] == "3.13.13", base_packages
assert base_packages["numpy"] == "2.4.6", base_packages
assert base_packages["pandas"] == "3.0.3", base_packages
assert base_packages["scikit-learn"] == "1.9.0", base_packages

base_specs = {name: importlib.util.find_spec(name) is not None for name in ["xgboost", "lightgbm", "catboost"]}
print("Base GBDT import availability:", base_specs)


{'python': '3.13.13', 'numpy': '2.4.6', 'pandas': '3.0.3', 'scikit-learn': '1.9.0', 'xgboost': 'not_installed', 'lightgbm': 'not_installed', 'catboost': 'not_installed'}
Base GBDT import availability: {'xgboost': False, 'lightgbm': False, 'catboost': False}


## Create Separate Wave 2 Environment and Install Pinned Packages


In [4]:
install_rows: list[dict] = []
dependency_rows: list[dict] = []

builder = venv.EnvBuilder(with_pip=True, clear=False, symlinks=False)
builder.create(ENV_DIR)
assert ENV_PYTHON.exists(), f"Missing separate env Python: {ENV_PYTHON}"

install_groups = [
    {
        "group": "core_plus_wave2a",
        "packages": [
            "numpy==2.4.6",
            "pandas==3.0.3",
            "scikit-learn==1.9.0",
            "xgboost==3.2.0",
            "lightgbm==4.6.0",
        ],
        "required": True,
    },
    {
        "group": "catboost_wave2b",
        "packages": ["catboost==1.2.10"],
        "required": False,
    },
]

for group in install_groups:
    args = [str(ENV_PYTHON), "-m", "pip", "install", *group["packages"]]
    result = run_cmd(args, cwd=REPO_ROOT, check=False)
    install_rows.append(
        {
            "group": group["group"],
            "packages": " ".join(group["packages"]),
            "returncode": result.returncode,
            "stdout_tail": result.stdout[-1000:].replace("\n", " | "),
            "stderr_tail": result.stderr[-1000:].replace("\n", " | "),
            "required": group["required"],
        }
    )
    if group["required"] and result.returncode != 0:
        break

core_install_ok = install_rows[0]["returncode"] == 0
if not core_install_ok:
    print("Required Wave 2A install failed; reports will be written and the notebook will stop.")


$ C:\tmp\reto_tokio_phase8_wave2_env\Scripts\python.exe -m pip install numpy==2.4.6 pandas==3.0.3 scikit-learn==1.9.0 xgboost==3.2.0 lightgbm==4.6.0
[pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   --------------------------------- ------ 10/12 [pandas]
   -------------

$ C:\tmp\reto_tokio_phase8_wave2_env\Scripts\python.exe -m pip install catboost==1.2.10
-- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly]
   ----------------------------- ----------  8/11 [plotly

## Separate Environment Import Verification


In [5]:
verification_code = r"""
import os
os.environ["MPLBACKEND"] = "Agg"
import importlib
import json
import platform
packages = ["numpy", "pandas", "sklearn", "xgboost", "lightgbm", "catboost"]
out = {"python": platform.python_version(), "packages": {}}
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "unknown")
        if pkg == "sklearn":
            version = mod.__version__
        out["packages"][pkg] = {"status": "ok", "version": version}
    except Exception as exc:
        out["packages"][pkg] = {"status": "failed", "version": "", "error": repr(exc)}
print(json.dumps(out, sort_keys=True))
"""
verify_result = run_cmd([str(ENV_PYTHON), "-c", verification_code], cwd=REPO_ROOT, check=False)
import json
verify_payload = json.loads(verify_result.stdout.strip().splitlines()[-1]) if verify_result.stdout.strip() else {
    "python": "unknown",
    "packages": {},
    "error": verify_result.stderr,
}

required_expected = {
    "numpy": "2.4.6",
    "pandas": "3.0.3",
    "sklearn": "1.9.0",
    "xgboost": "3.2.0",
    "lightgbm": "4.6.0",
}
optional_expected = {"catboost": "1.2.10"}

for package, expected in required_expected.items():
    info = verify_payload["packages"].get(package, {})
    dependency_rows.append(
        {
            "package": package,
            "required_version": expected,
            "installed_version": info.get("version", ""),
            "import_status": info.get("status", "missing"),
            "install_status": "ok" if info.get("status") == "ok" and info.get("version") == expected else "failed",
            "required": "yes",
            "notes": info.get("error", ""),
        }
    )

for package, expected in optional_expected.items():
    info = verify_payload["packages"].get(package, {})
    dependency_rows.append(
        {
            "package": package,
            "required_version": expected,
            "installed_version": info.get("version", ""),
            "import_status": info.get("status", "missing"),
            "install_status": "ok" if info.get("status") == "ok" and info.get("version") == expected else "failed_or_omitted",
            "required": "conditional",
            "notes": info.get("error", "CatBoost omitted if gate/import fails."),
        }
    )

required_imports_ok = all(row["install_status"] == "ok" for row in dependency_rows if row["required"] == "yes")
catboost_ok = any(row["package"] == "catboost" and row["install_status"] == "ok" for row in dependency_rows)
print("Required imports OK:", required_imports_ok)
print("CatBoost gate import OK:", catboost_ok)


$ C:\tmp\reto_tokio_phase8_wave2_env\Scripts\python.exe -c 
import os
os.environ["MPLBACKEND"] = "Agg"
import importlib
import json
import platform
packages = ["numpy", "pandas", "sklearn", "xgboost", "lightgbm", "catboost"]
out = {"python": platform.python_version(), "packages": {}}
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "unknown")
        if pkg == "sklearn":
            version = mod.__version__
        out["packages"][pkg] = {"status": "ok", "version": version}
    except Exception as exc:
        out["packages"][pkg] = {"status": "failed", "version": "", "error": repr(exc)}
print(json.dumps(out, sort_keys=True))

{"packages": {"catboost": {"status": "ok", "version": "1.2.10"}, "lightgbm": {"status": "ok", "version": "4.6.0"}, "numpy": {"status": "ok", "version": "2.4.6"}, "pandas": {"status": "ok", "version": "3.0.3"}, "sklearn": {"status": "ok", "version": "1.9.0"}, "xgboost": {"status": "ok", "versio

## Write Dependency and Environment Reports


In [6]:
dependency_fieldnames = [
    "package",
    "required_version",
    "installed_version",
    "import_status",
    "install_status",
    "required",
    "notes",
]
safe_write_csv(DEPENDENCY_REPORT, dependency_rows, dependency_fieldnames)

install_table = "\n".join(
    f"- {row['group']}: returncode={row['returncode']}; required={row['required']}"
    for row in install_rows
)
dependency_table = "\n".join(
    f"| {row['package']} | {row['required_version']} | {row['installed_version']} | {row['import_status']} | {row['install_status']} | {row['required']} |"
    for row in dependency_rows
)
report = f"""# Phase 8 Wave 2 Environment Report

## Scope
Separate environment dependency report for `phase8_wave2_external_gbdt_v1`.

## Repository
- Authorized HEAD: `{AUTHORIZED_HEAD}`
- Observed HEAD: `{head}`
- Repository root: `{REPO_ROOT}`

## Base Environment
- Base Python executable: `{sys.executable}`
- Base Python: `{base_packages['python']}`
- Base numpy: `{base_packages['numpy']}`
- Base pandas: `{base_packages['pandas']}`
- Base scikit-learn: `{base_packages['scikit-learn']}`
- Base GBDT availability: `{base_specs}`

## Separate Wave 2 Environment
- Path: `{ENV_DIR}`
- Python: `{verify_payload.get('python', 'unknown')}`
- Required imports OK: `{required_imports_ok}`
- CatBoost import OK: `{catboost_ok}`

## Install Groups
{install_table}

## Dependency Table
| package | required_version | installed_version | import_status | install_status | required |
|---|---:|---:|---|---|---|
{dependency_table}

## Guardrails
- The existing project `.venv` was not installed into or modified by this notebook.
- `requirements.txt`, lockfiles, `logs/experiment_log.csv`, official data, references, and submissions were not modified by this notebook.
- Phase 9, Phase 10, and Phase 11 remain locked.
"""
safe_write_text(ENVIRONMENT_REPORT, textwrap.dedent(report).strip() + "\n")

print("Wrote:", DEPENDENCY_REPORT)
print("Wrote:", ENVIRONMENT_REPORT)
if not required_imports_ok:
    raise RuntimeError("Required Wave 2A dependency/import gate failed. Stop before model comparison.")


Wrote: C:\GitHub\reto_Tokio\outputs\validation\phase8_wave2_external_gbdt_v1_dependency_report.csv
Wrote: C:\GitHub\reto_Tokio\outputs\reports\phase8_wave2_external_gbdt_v1_environment_report.md
